# 00 Setup and tour

Welcome to Hudhud. Every notebook in this folder follows the same rhythm:

1. **Concept.** A short explanation of the idea.
2. **Your code.** Cells tagged `stub` contain signatures and docstrings with
   `raise NotImplementedError`. You replace the bodies.
3. **Tests.** Cells that start with `%%ipytest` run pytest on the code above.
   Red means keep going; green means done.
4. **Graduate.** From notebook 01 on, the finished code moves into
   `hub/src/hudhud_hub/<module>.py` and the tests move into `hub/tests/`.

Reference answers live in `notebooks/solutions/`. Open them only when stuck.

## The project layout

- `hub/` is a uv project. `uv sync --project hub` builds its virtual
  environment, and `uv run --project hub <cmd>` runs a command inside it. You
  started this Jupyter server that way, so this kernel already has every
  dependency.
- `hub/src/hudhud_hub/` is the package the server will import. It is almost
  empty on purpose: you fill it in, one notebook at a time.
- `hub/tests/` holds the graduated tests, plus a checker that runs every
  notebook against the reference answers.

## SQLite from Python

The hub keeps everything in one SQLite file. Python ships `sqlite3` in the
standard library. Three habits matter from day one:

- **Always use placeholders** (`?`), never f-strings, for values. Placeholders
  stop SQL injection and handle quoting for you, including Farsi text.
- `conn.execute(...)` returns a cursor. `cursor.lastrowid` is the id of the row
  you just inserted.
- An in-memory database (`":memory:"`) is perfect for tests: fast, and gone
  when the connection closes.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
import os
import sqlite3

import ipytest

ipytest.autoconfig(raise_on_error=os.environ.get("HUDHUD_NB_CHECK") == "1")

## Your code

A tiny warm-up: a `books` table with an integer id, a required title and an
optional author.

In [ ]:
def create_books_table(conn: sqlite3.Connection) -> None:
    """Create table `books(id INTEGER PRIMARY KEY, title TEXT NOT NULL, author TEXT)`."""
    raise NotImplementedError


def add_book(conn: sqlite3.Connection, title: str, author: str | None) -> int:
    """Insert one book with a parameterised query and return its new id."""
    raise NotImplementedError


def titles_by_author(conn: sqlite3.Connection, author: str) -> list[str]:
    """Titles by exactly this author, sorted alphabetically."""
    raise NotImplementedError

## Tests

In [ ]:
%%ipytest -q
import pytest


def fresh():
    conn = sqlite3.connect(":memory:")
    create_books_table(conn)
    return conn


def test_add_book_returns_increasing_ids():
    conn = fresh()
    first = add_book(conn, "Masnavi", "Rumi")
    second = add_book(conn, "Divan", "Hafez")
    assert second > first


def test_titles_by_author_sorted():
    conn = fresh()
    add_book(conn, "Masnavi", "Rumi")
    add_book(conn, "Fihi Ma Fihi", "Rumi")
    add_book(conn, "Divan", "Hafez")
    assert titles_by_author(conn, "Rumi") == ["Fihi Ma Fihi", "Masnavi"]


def test_farsi_text_round_trips():
    conn = fresh()
    add_book(conn, "منطق‌الطیر", "عطار")
    assert titles_by_author(conn, "عطار") == ["منطق‌الطیر"]


def test_author_is_a_value_not_sql():
    conn = fresh()
    add_book(conn, "Masnavi", "Rumi")
    assert titles_by_author(conn, "x' OR '1'='1") == []


def test_title_is_required():
    conn = fresh()
    with pytest.raises(sqlite3.IntegrityError):
        add_book(conn, None, "Nobody")

## Done?

When every test is green you have the rhythm. Notebook 01 is the first one
whose code graduates into the hub.